# Defter 22 — Figures for the manuscript (Taslak 11)
**Theory-guided AI reveals contrasting effects of a century of climate change on tree growth: slower in dry forests, not in cold ones** · Prof. Dr. İlker Ercanlı

**Ne yapar (TR):** Yeni hesap yok. Defter 20 (`SONUC_20`), Defter 21 (`SONUC_21`) ve taslak 10'un halka tabanlı analiz tablolarından
(`ANALIZ_SON`) makalenin 6 ana şeklini ve Extended Data 2–5'i Nature biçiminde çizer (89 / 183 mm genişlik, 5–7 pt yazı, PDF + 300 dpi PNG).
Her şeklin altındaki sayılar ayrıca `SOURCE_DATA_FigN.csv` olarak kaydedilir (Nature "Source Data" ister).

| Şekil | İçerik | Kaynak tablolar |
|---|---|---|
| Fig. 1 | Isınma ve toprak nemi değişimi (harita), on yıllık sıcaklık anomalisi | `Q0_iklim_degisimi_alan`, `Q0_iklim_degisimi_onyil` |
| Fig. 2 | 1990–2019 atfedilen büyüme değişimi (harita), gruplar: ağ ve halka tahmini | `Q3_atif_alan`, `Q3_atif_grup`, `P2b_ana_blok_araliklari` |
| Fig. 3 | On yıllara göre ortaya çıkış (sabit saha kümesi) ve saha sayısı | `Q3_atif_onyil` |
| Fig. 4 | Kanal payları; sınırlayıcı faktör payları | `Q3_atif_kanal`, `Q4_sinirlayici_grup` |
| Fig. 5 | Referans ağacın 100 yıl sonraki çap farkı ve 15 cm'ye ulaşma yaşı | `Q5_buyume_egrisi_grup` |
| Fig. 6 | Başarı ve belirsizlik | `P3_ana_kosu`, `P8_saha_testi`, `P5_topluluk_belirsizlik`, `Q1_saha_kalibrasyon`, `P6_belirsizlik_ondalik`, `P7_yillik_belirsizlik` |
| ED2 | Mimari arama | `P2_mimari_arama`, `P2_secilen_mimari.json` |
| ED3 | Yüzyıllık artık eğilimi | `Q2_sapma_yas_onyil`, `P7_yillik_belirsizlik` |
| ED4 | Halka tabanlı tahmin: sıcak yıl sıklığı, sağlamlık, blok boyutu | `M4_uc_olay_siklik`, `P2_saglamlik_blok`, `Q3b_blok_boyutu` |
| ED5 | Cinslere göre atıf: ağ ve halka tahmini | `Q3_atif_grup`, `P2b_ana_blok_araliklari` |

ED1 (ağın yapısı) bir çizim olduğu için burada üretilmez; taslak 10'un ED1'i VPD, sıcaklık optimumu ve ağaç etkisi eklenerek elle güncellenir.

**Çalışma zamanı:** CPU yeterli (GPU gerekmez) → "Tümünü çalıştır". Haritalar için `cartopy` kurulur; kurulamazsa haritalar kıyı çizgisi olmadan çizilir.
**Çıktılar:** `ITRDB_AI/SEKILLER_22/`. **Bitiş satırı:** `==== DEFTER 22 TAMAM ====`

In [ ]:
AYAR = dict(
    kok='/content/drive/MyDrive/ITRDB_AI',      # project folder on Drive
    sonuc20='SONUC_20', sonuc21='SONUC_21',
    analiz_son=None,                             # folder of the draft-10 ring-based tables; None = search for 'ANALIZ_SON'
    cikti='SEKILLER_22',
    harita_kiyi=True,                            # draw coastlines with cartopy if available
    yerel_deneme=False,                          # True only for a local test with synthetic tables
)

In [ ]:
import os, sys, glob, json, subprocess, warnings
warnings.filterwarnings('ignore')
if not AYAR['yerel_deneme']:
    from google.colab import drive
    drive.mount('/content/drive')
    if AYAR['harita_kiyi']:
        try:
            import cartopy  # noqa: F401
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cartopy'], check=False)
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

K20 = os.path.join(AYAR['kok'], AYAR['sonuc20']); K21 = os.path.join(AYAR['kok'], AYAR['sonuc21'])
if AYAR['analiz_son'] is None:
    aday = glob.glob(os.path.join(AYAR['kok'], '**', 'ANALIZ_SON'), recursive=True) or \
           glob.glob('/content/drive/MyDrive/**/ANALIZ_SON', recursive=True)
    KAS = aday[0] if aday else None
else:
    KAS = AYAR['analiz_son']
CIKTI = os.path.join(AYAR['kok'], AYAR['cikti']); os.makedirs(CIKTI, exist_ok=True)
print('SONUC_20  :', K20, os.path.isdir(K20)); print('SONUC_21  :', K21, os.path.isdir(K21))
print('ANALIZ_SON:', KAS); print('Output    :', CIKTI)

def oku(klasor, ad, zorunlu=True):
    yol = os.path.join(klasor, ad) if klasor else None
    if yol and os.path.exists(yol):
        return pd.read_csv(yol)
    if zorunlu: raise FileNotFoundError(f'{ad} not found in {klasor}')
    print(f'  (missing, panel skipped: {ad})'); return None

In [ ]:
# --- Nature style: Arial/Helvetica (Liberation Sans as metric-compatible fallback), 5-7 pt, thin lines ---
from matplotlib import font_manager
aile = next((f for f in ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'] if any(f == x.name for x in font_manager.fontManager.ttflist)), 'DejaVu Sans')
mpl.rcParams.update({'font.family': aile, 'font.size': 6, 'axes.titlesize': 6.5, 'axes.labelsize': 6, 'xtick.labelsize': 5.5,
                     'ytick.labelsize': 5.5, 'legend.fontsize': 5.5, 'axes.linewidth': 0.5, 'xtick.major.width': 0.5,
                     'ytick.major.width': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2, 'lines.linewidth': 1.0,
                     'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'ps.fonttype': 42,
                     'savefig.dpi': 300, 'figure.dpi': 150, 'legend.frameon': False})
MM = 1 / 25.4; TEK, CIFT = 89 * MM, 183 * MM
# group colours (colour-blind safe; also distinguished by marker/line style)
RENK = {'All sites': '#3b3b3b', 'Driest 25%': '#b35806', 'Coldest 25%': '#2166ac', 'Region: Europe': '#7b3294',
        'Wettest 25%': '#1b7837'}
ETIKET = {'All sites': 'All sites', 'Driest 25%': 'Driest quarter', 'Coldest 25%': 'Coldest quarter', 'Wettest 25%': 'Wettest quarter',
          'Region: Europe': 'Europe', 'Region: North America': 'North America', 'Region: Asia': 'Asia',
          'Region: Southern Hemisphere': 'Southern Hemisphere'}
CIZGI = {'All sites': '-', 'Driest 25%': '-', 'Coldest 25%': '--', 'Region: Europe': ':'}
def etiket(g): return ETIKET.get(g, g.replace('Genus: ', ''))
def harf(ax, h, x=-0.16, y=1.04):
    ax.text(x, y, h, transform=ax.transAxes, fontsize=8, fontweight='bold', va='bottom', ha='left')
def kaydet(fig, ad, kaynak=None):
    for uz in ('pdf', 'png'):
        fig.savefig(os.path.join(CIKTI, f'{ad}.{uz}'), bbox_inches='tight')
    if kaynak is not None:
        kaynak.to_csv(os.path.join(CIKTI, f'SOURCE_DATA_{ad}.csv'), index=False)
    print('saved', ad); plt.close(fig)

try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature, cartopy.io.shapereader as shpreader
    KARTOPI = AYAR['harita_kiyi']
    if KARTOPI:   # coastlines are downloaded on first use; check now so that saving a figure cannot fail later
        for ad_ in ('coastline', 'land'): shpreader.natural_earth(resolution='110m', category='physical', name=ad_)
except Exception as e:
    print('cartopy coastlines unavailable, maps drawn without them:', str(e)[:80])
    KARTOPI = False
print('font:', aile, '| cartopy:', KARTOPI)

def harita_ekseni(fig, konum):
    if KARTOPI:
        ax = fig.add_axes(konum, projection=ccrs.Robinson())
        ax.set_extent([-180, 180, -57, 84], crs=ccrs.PlateCarree())
        try:
            ax.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#ececec', edgecolor='none', zorder=0)
            ax.add_feature(cfeature.COASTLINE.with_scale('110m'), linewidth=0.25, edgecolor='#9a9a9a', zorder=1)
        except Exception as e:
            print('  coastline not available:', e)
        ax.spines['geo'].set_linewidth(0.4)
        return ax, dict(transform=ccrs.PlateCarree())
    ax = fig.add_axes(konum); ax.set_xlim(-180, 180); ax.set_ylim(-58, 82); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([]); [s.set_visible(True) for s in ax.spines.values()]
    ax.set_facecolor('#f4f4f4')
    return ax, {}

def harita(fig, konum, lon, lat, deger, cmap, vmin, vmax, baslik, birim, isaret=None, harf_=None):
    """Site map. isaret: boolean mask of confident sites; these are drawn larger and opaque, the others small and faded."""
    ax, tr = harita_ekseni(fig, konum)
    lon, lat, deger = np.asarray(lon), np.asarray(lat), np.asarray(deger)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0, rasterized=True, **tr)
    if isaret is None:
        o = np.argsort(np.abs(deger)); sc = ax.scatter(lon[o], lat[o], c=deger[o], s=1.6, zorder=3, **kw)
    else:
        z = ~isaret; ax.scatter(lon[z], lat[z], c=deger[z], s=0.8, alpha=0.45, zorder=3, **kw)
        o = np.where(isaret)[0]; o = o[np.argsort(np.abs(deger[o]))]
        sc = ax.scatter(lon[o], lat[o], c=deger[o], s=2.4, zorder=4, **kw)
    ax.text(0.0, 1.015, (harf_ + '   ') if harf_ else '', transform=ax.transAxes, fontsize=8, fontweight='bold', va='bottom')
    ax.text(0.045 if harf_ else 0.0, 1.015, baslik, transform=ax.transAxes, fontsize=6.5, va='bottom')
    cax = fig.add_axes([konum[0] + konum[2] * 0.30, konum[1] - 0.02, konum[2] * 0.40, 0.014])
    cb = fig.colorbar(sc, cax=cax, orientation='horizontal', extend='both'); cb.outline.set_linewidth(0.3)
    cb.ax.tick_params(labelsize=5, width=0.4, length=1.5, pad=1); cb.set_label(birim, fontsize=5.5, labelpad=1)
    return ax

In [ ]:
# --- Load tables ---
Q0a = oku(K21, 'Q0_iklim_degisimi_alan.csv'); Q0o = oku(K21, 'Q0_iklim_degisimi_onyil.csv')
Q3a = oku(K21, 'Q3_atif_alan.csv'); Q3g = oku(K21, 'Q3_atif_grup.csv'); Q3o = oku(K21, 'Q3_atif_onyil.csv')
Q3k = oku(K21, 'Q3_atif_kanal.csv'); Q4g = oku(K21, 'Q4_sinirlayici_grup.csv')
Q5c = oku(K21, 'Q5_egri_ortalama.csv'); Q5g = oku(K21, 'Q5_buyume_egrisi_grup.csv')
Q1 = oku(K21, 'Q1_saha_kalibrasyon.csv'); Q2y = oku(K21, 'Q2_sapma_yas_onyil.csv', zorunlu=False)
P2 = oku(K20, 'P2_mimari_arama.csv', zorunlu=False); P3 = oku(K20, 'P3_ana_kosu.csv'); P5 = oku(K20, 'P5_topluluk_belirsizlik.csv')
P6 = oku(K20, 'P6_belirsizlik_ondalik.csv'); P7 = oku(K20, 'P7_yillik_belirsizlik.csv'); P8 = oku(K20, 'P8_saha_testi.csv')
SECIM = json.load(open(os.path.join(K20, 'P2_secilen_mimari.json'))) if os.path.exists(os.path.join(K20, 'P2_secilen_mimari.json')) else dict(gizli=128, katman=2)
HB = oku(KAS, 'P2b_ana_blok_araliklari.csv', zorunlu=False)     # ring-based estimate, 10-degree spatial-block intervals
HS = oku(KAS, 'P2_saglamlik_blok.csv', zorunlu=False)           # robustness specifications
HF = oku(KAS, 'M4_uc_olay_siklik.csv', zorunlu=False)           # extreme-year frequency
HK = oku(KAS, 'Q3b_blok_boyutu.csv', zorunlu=False)             # block size sensitivity
ANA = 'teori_vpd_opt_re'; IKLIMSIZ = 'iklimsiz_re'
DON = '1990-2019'
# ring-based group names (Turkish in the draft-10 tables) -> Defter 21 group names
TR2EN = {'Tüm alanlar': 'All sites', 'En soğuk %25': 'Coldest 25%', 'En kurak %25': 'Driest 25%'}
def tr2en(g):
    g = TR2EN.get(g, g)
    return g.replace('Bölge: ', 'Region: ').replace('Cins: ', 'Genus: ')
if HB is not None: HB['group'] = HB['grup'].map(tr2en)
print('tables loaded')

In [ ]:
# --- Fig. 1 | Climate change at the tree-ring sites ---
fig = plt.figure(figsize=(CIFT, 0.56 * CIFT))
d = Q0a[Q0a.cc_dT_C.notna()]
axa = harita(fig, [0.0, 0.60, 0.50, 0.34], d.lon, d.lat, d.cc_dT_C, 'YlOrRd', 0, 2.5,
             'Warming attributable to climate change, 1990–2019', '°C', harf_='a')
axb = harita(fig, [0.0, 0.12, 0.50, 0.34], d.lon, d.lat, d.cc_dM, 'BrBG', -0.03, 0.03,
             'Attributable change in soil moisture, 1990–2019', 'v/v', harf_='b')
axc = fig.add_axes([0.60, 0.10, 0.38, 0.82])
src1 = []
for g in ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe']:
    x = Q0o[Q0o.group == g].sort_values('decade')
    xc = x.decade + 5
    axc.plot(xc, x.cru_dT_C, CIZGI[g], color=RENK[g], label=etiket(g), marker='o', ms=2)
    src1.append(x.assign(panel='c'))
x = Q0o[Q0o.group == 'All sites'].sort_values('decade')
x = x[x.decade <= 2010]   # ISIMIP3a counterfactual ends in 2019
axc.plot(x.decade + 5, x.cc_dT_C, color='#888888', lw=0.8, ls=(0, (1, 1)), label='Climate-change signal (all sites)')
axc.axhline(0, color='k', lw=0.4); axc.set_xlim(1900, 2026)
axc.set_ylabel('Temperature anomaly vs 1901–1960 (°C)'); axc.set_xlabel('Decade')
axc.legend(loc='upper left'); harf(axc, 'c')
kaydet(fig, 'Fig1', pd.concat([d[['studyCode', 'lat', 'lon', 'region', 'cc_dT_C', 'cc_dM', 'cru_dT_C']].assign(panel='a,b'),
                               pd.concat(src1)], ignore_index=True))

In [ ]:
# --- Fig. 2 | Growth change attributed to climate change, 1990-2019 ---
kol = f'total {DON} (%)'; p05 = f'total {DON} members p05 (%)'; p95 = f'total {DON} members p95 (%)'
d = Q3a[Q3a[kol].notna()].copy()
emin = ((d[p95] < 0) | (d[p05] > 0)).to_numpy()
fig = plt.figure(figsize=(CIFT, 0.38 * CIFT))
axa = harita(fig, [0.0, 0.16, 0.58, 0.76], d.lon, d.lat, d[kol].clip(-10, 10), 'BrBG', -6, 6,
             f'Growth change attributed to climate change, 1990–2019 ({len(d):,} sites)', '% per year', isaret=emin, harf_='a')
axb = fig.add_axes([0.72, 0.12, 0.27, 0.78])
gr = ['Driest 25%', 'Region: Europe', 'All sites', 'Region: North America', 'Region: Asia', 'Coldest 25%']
yy = np.arange(len(gr))[::-1]; src2 = []
for y, g in zip(yy, gr):
    r = Q3g[(Q3g.period == DON) & (Q3g.channel == 'total') & (Q3g.group == g)]
    if len(r):
        r = r.iloc[0]
        axb.errorbar(r.median_pct, y + 0.15, xerr=[[r.median_pct - r.boot_lo95], [r.boot_hi95 - r.median_pct]], fmt='o', ms=3,
                     color='#1f78b4', elinewidth=0.7, capsize=0)
        src2.append(dict(group=g, source='network', estimate=r.median_pct, lo95=r.boot_lo95, hi95=r.boot_hi95, n_sites=r.n_sites))
    if HB is not None:
        h = HB[(HB.secim == 'Ana') & (HB.group == g)]
        if len(h):
            h = h.iloc[0]
            axb.errorbar(h.toplam_yuzde_yil, y - 0.15, xerr=[[h.toplam_yuzde_yil - h.blok_alt95], [h.blok_ust95 - h.toplam_yuzde_yil]],
                         fmt='D', ms=2.6, color='#e66101', elinewidth=0.7, capsize=0)
            src2.append(dict(group=g, source='rings (model-free)', estimate=h.toplam_yuzde_yil, lo95=h.blok_alt95, hi95=h.blok_ust95, n_sites=h.n_alan))
axb.axvline(0, color='k', lw=0.4); axb.set_yticks(yy); axb.set_yticklabels([etiket(g) for g in gr])
axb.set_xlabel('Growth change, 1990–2019 (% per year)'); axb.set_ylim(-0.6, len(gr) - 0.4)
axb.legend(handles=[Line2D([], [], marker='o', color='#1f78b4', ls='', ms=3, label='Network (20 members)'),
                    Line2D([], [], marker='D', color='#e66101', ls='', ms=2.6, label='Rings (no network)')],
           loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=1)
harf(axb, 'b', x=-0.55)
kaydet(fig, 'Fig2', pd.concat([d[['studyCode', 'lat', 'lon', 'region', 'genus', kol, p05, p95]].assign(panel='a'),
                               pd.DataFrame(src2).assign(panel='b')], ignore_index=True))

In [ ]:
# --- Fig. 3 | Emergence of the divide (fixed set of sites) ---
fig, ax = plt.subplots(figsize=(TEK, 0.78 * TEK))
x0 = Q3o[(Q3o.channel == 'total') & (Q3o.site_set == 'fixed_set')]
src3 = []
for g in ['Coldest 25%', 'All sites', 'Region: Europe', 'Driest 25%']:
    x = x0[x0.group == g].sort_values('decade')
    if not len(x): continue
    xc = x.decade + 5
    ax.fill_between(xc, x.members_p05, x.members_p95, color=RENK[g], alpha=0.18, lw=0)
    ax.plot(xc, x.median_pct, CIZGI[g], color=RENK[g], marker='o', ms=2.2, label=etiket(g))
    src3.append(x)
ax.axhline(0, color='k', lw=0.4); ax.set_xlim(1900, 2020)
ax.set_xlabel('Decade'); ax.set_ylabel('Growth change attributed to climate change (% per year)')
ax.legend(loc='lower left', ncol=2)
# number of sites per decade (all sites of the fixed set) along the top
na = x0[x0.group == 'All sites'].sort_values('decade')
for dc, n in zip(na.decade, na.n_sites):
    ax.text(dc + 5, 1.0, f'{n:,}', transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=4.5, color='#555555', rotation=90)
ax.text(1.0, 1.0, ' n', transform=ax.transAxes, ha='left', va='bottom', fontsize=4.5, color='#555555')
kaydet(fig, 'Fig3', pd.concat(src3, ignore_index=True))

In [ ]:
# --- Fig. 4 | Mechanisms: climatic channels and limiting factors ---
fig = plt.figure(figsize=(CIFT, 0.40 * CIFT))
axa = fig.add_axes([0.06, 0.16, 0.40, 0.74])
grk = ['All sites', 'Coldest 25%', 'Driest 25%', 'Region: Europe', 'Genus: Fagus', 'Genus: Cedrus']
kan = [('temperature', 'Temperature', '#d6604d'), ('soil moisture', 'Soil moisture', '#8c510a'), ('VPD', 'VPD', '#5aae61'),
       ('interaction', 'Interaction (law of the minimum)', '#999999')]
k = Q3k[Q3k.period == DON].set_index('group'); w = 0.19; xg = np.arange(len(grk))
for j, (c, ad, rk) in enumerate(kan):
    v = [k.loc[g, c] if g in k.index else np.nan for g in grk]
    axa.bar(xg + (j - 1.5) * w, v, w, color=rk, label=ad, lw=0)
tot = [k.loc[g, 'total'] if g in k.index else np.nan for g in grk]
axa.scatter(xg, tot, marker='_', s=60, color='k', lw=1.2, zorder=5, label='Total')
axa.axhline(0, color='k', lw=0.4); axa.set_xticks(xg); axa.set_xticklabels([etiket(g).replace(' quarter', '\nquarter') for g in grk])
axa.set_ylabel('Attributed change, 1990–2019 (% per year)'); axa.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3); harf(axa, 'a', x=-0.13)
src4 = k.loc[[g for g in grk if g in k.index]].reset_index().assign(panel='a')
# b: shares of the growing season limited by each factor, by period
fak = [('VPD', 'VPD', '#5aae61'), ('heat', 'Heat (above optimum)', '#d6604d'), ('soil moisture', 'Soil moisture', '#8c510a'),
       ('cold', 'Cold', '#4393c3')]          # VPD at the base so that its rise reads from zero
gb = ['All sites', 'Region: Europe', 'Genus: Fagus']; per = ['1902-1930', '1931-1960', '1961-1990', '1991-2019']
axb = fig.add_axes([0.56, 0.16, 0.43, 0.74]); pos, lab = [], []; src4b = []
for i, g in enumerate(gb):
    for j, p in enumerate(per):
        r = Q4g[(Q4g.group == g) & (Q4g.period == p)]
        if not len(r): continue
        r = r.iloc[0]; xpos = i * 5.2 + j; alt = 0
        tot_ = sum(r[f] for f, _, _ in fak)
        for f, ad, rk in fak:
            v = r[f] / tot_; axb.bar(xpos, v, 0.85, bottom=alt, color=rk, lw=0); alt += v
        pos.append(xpos); lab.append(p.replace('-', '–\n')); src4b.append(r.to_dict())
    axb.text(i * 5.2 + 1.5, 1.04, etiket(g), ha='center', va='bottom', fontsize=6, transform=axb.get_xaxis_transform())
axb.set_xticks(pos); axb.set_xticklabels(lab, fontsize=4.6); axb.set_ylim(0, 1); axb.set_ylabel('Share of the growing season')
axb.legend(handles=[Patch(color=rk, label=ad) for _, ad, rk in fak][::-1], loc='upper left', bbox_to_anchor=(1.0, 1.0))
harf(axb, 'b', x=-0.10)
kaydet(fig, 'Fig4', pd.concat([src4, pd.DataFrame(src4b).assign(panel='b')], ignore_index=True))

In [ ]:
# --- Fig. 5 | The shift accumulates over a tree's life ---
fig = plt.figure(figsize=(CIFT * 0.78, 0.36 * CIFT))
F, C = 'factual 1920-2019', 'counterfactual 1920-2019'
# a: median over sites of the per-site diameter difference after 100 years (the quantity reported in the text);
#    a ratio of median curves would mix different sites at each age and is not shown
axb = fig.add_axes([0.17, 0.17, 0.36, 0.74]); src5 = []
VS = f'{F} vs {C}'
g5 = ['Genus: Larix', 'Coldest 25%', 'All sites', 'Driest 25%', 'Region: Europe', 'Genus: Pseudotsuga', 'Genus: Fagus', 'Genus: Cedrus']
g5 = [g for g in g5 if ((Q5g.group == g) & (Q5g.scenario == VS)).any()]
for i, g in enumerate(g5):
    r = Q5g[(Q5g.group == g) & (Q5g.scenario == VS)].iloc[0]; v = float(r.median_diameter_end)
    rk = RENK.get(g, '#1b7837' if v > 0 else '#8c510a')
    axb.plot([0, v], [i, i], color=rk, lw=1.4, solid_capstyle='butt'); axb.scatter([v], [i], color=rk, s=12, zorder=3)
    axb.text(v + (0.25 if v >= 0 else -0.25), i, f'{v:+.1f}%'.replace('-', '−'), va='center', ha='left' if v >= 0 else 'right', fontsize=5)
    src5.append(pd.DataFrame(dict(group=[g], diameter_difference_100yr_pct=[v], share_negative=[r.get('share_negative', np.nan)], n_sites=[r.n_sites])))
axb.set_yticks(range(len(g5))); axb.set_yticklabels([etiket(g) for g in g5]); axb.invert_yaxis(); axb.axvline(0, color='k', lw=0.4)
lim = max(abs(float(Q5g[(Q5g.scenario == VS) & Q5g.group.isin(g5)].median_diameter_end.abs().max())) * 1.35, 2)
axb.set_xlim(-lim, lim); axb.set_xlabel('Stem diameter after 100 years,\nobserved vs without climate change (%)')
harf(axb, 'a', x=-0.55)
axc = fig.add_axes([0.73, 0.17, 0.26, 0.74])
e1, e2 = 'climate 1931-1960 repeated', 'climate 1990-2019 repeated'; kol15 = 'median_years_to_15cm'
gc = ['Genus: Fagus', 'Genus: Cedrus', 'Driest 25%', 'Coldest 25%', 'Genus: Larix']
if kol15 in Q5g.columns:
    for i, g in enumerate(gc):
        a = Q5g[(Q5g.group == g) & (Q5g.scenario == e1)]; b = Q5g[(Q5g.group == g) & (Q5g.scenario == e2)]
        if not (len(a) and len(b)): continue
        a, b = float(a[kol15].iloc[0]), float(b[kol15].iloc[0])
        axc.plot([a, b], [i, i], color='#bbbbbb', lw=1.2, zorder=1)
        axc.scatter([a], [i], color='#92c5de', s=10, zorder=2); axc.scatter([b], [i], color='#b2182b', s=10, zorder=3)
        src5.append(pd.DataFrame(dict(group=[g], years_to_15cm_1931_1960=[a], years_to_15cm_1990_2019=[b])))
    axc.set_yticks(range(len(gc))); axc.set_yticklabels([etiket(g).replace(' quarter', '') for g in gc]); axc.invert_yaxis()
    axc.set_xlabel('Age at 15 cm diameter (years)')
    axc.legend(handles=[Line2D([], [], marker='o', ls='', color='#92c5de', ms=3, label='Climate of 1931–1960'),
                        Line2D([], [], marker='o', ls='', color='#b2182b', ms=3, label='Climate of 1990–2019')],
               loc='lower right', bbox_to_anchor=(1.0, 1.0))
harf(axc, 'b', x=-0.45)
kaydet(fig, 'Fig5', pd.concat(src5, ignore_index=True))

In [ ]:
# --- Fig. 6 | Skill and uncertainty ---
fig = plt.figure(figsize=(CIFT, 0.36 * CIFT)); src6 = []
axa = fig.add_axes([0.05, 0.18, 0.19, 0.72]); rng = np.random.default_rng(1)
kosul = [('Withheld years', P3, ANA, '#1f78b4'), ('Withheld years', P3, IKLIMSIZ, '#bdbdbd'),
         ('Withheld regions', P8, ANA, '#1f78b4'), ('Withheld regions', P8, IKLIMSIZ, '#bdbdbd')]
for i, (ad, tb, m, rk) in enumerate(kosul):
    v = tb[tb.model == m].blok_ici_r_medyan.to_numpy()
    xp = i + (i >= 2) * 0.5
    axa.scatter(xp + rng.uniform(-0.15, 0.15, len(v)), v, s=4, color=rk, lw=0, alpha=0.9)
    axa.plot([xp - 0.25, xp + 0.25], [v.mean()] * 2, color='k', lw=0.8)
    src6.append(pd.DataFrame(dict(panel='a', test=ad, model=m, within_block_r=v)))
axa.set_xticks([0.5, 3.0]); axa.set_xticklabels(['Withheld\nyears', 'Withheld\nregions']); axa.set_ylim(0, None)
axa.set_ylabel('Within-block correlation (median site)')
axa.legend(handles=[Line2D([], [], marker='o', ls='', color='#1f78b4', ms=3, label='With climate'),
                    Line2D([], [], marker='o', ls='', color='#bdbdbd', ms=3, label='Without climate')], loc='center right')
harf(axa, 'a', x=-0.30)
axb = fig.add_axes([0.32, 0.18, 0.17, 0.72])
p5 = P5[P5.model == ANA].iloc[0]; q1b = Q1[Q1.version == 'before']; q1a = Q1[Q1.version == 'after']
cub = [('Withheld\nyears', p5.kapsama90_toplam, '#1f78b4'), ('Unseen sites\n(no tree\nspread)', q1b.coverage90.mean(), '#fdb863'),
       ('Unseen sites\n(with tree\nspread)', q1a.coverage90.mean(), '#e66101')]
for i, (ad, v, rk) in enumerate(cub): axb.bar(i, v, 0.65, color=rk, lw=0)
axb.scatter(np.repeat(1, len(q1b)), q1b.coverage90, s=3, color='k', zorder=3); axb.scatter(np.repeat(2, len(q1a)), q1a.coverage90, s=3, color='k', zorder=3)
axb.axhline(0.9, color='k', lw=0.5, ls='--'); axb.set_ylim(0.5, 1.0); axb.set_xticks(range(3)); axb.set_xticklabels([c[0] for c in cub], fontsize=4.8)
axb.set_ylabel('Coverage of 90% prediction intervals'); harf(axb, 'b', x=-0.42)
src6.append(pd.DataFrame(dict(panel='b', test=[c[0].replace('\n', ' ') for c in cub], coverage90=[c[1] for c in cub])))
axc = fig.add_axes([0.57, 0.18, 0.17, 0.72])
p6 = P6[P6.model == ANA].sort_values('ondalik')
axc.plot(p6.ondalik + 1, p6.rmse_log, 'o-', color='#1f78b4', ms=2.5)
axc.set_xlabel('Decile of epistemic uncertainty'); axc.set_ylabel('RMSE of log ring width'); axc.set_xticks([1, 5, 10])
harf(axc, 'c', x=-0.36); src6.append(p6.assign(panel='c'))
axd = fig.add_axes([0.81, 0.18, 0.18, 0.72])
p7 = P7[(P7.model == ANA) & (P7.n >= 1000)].sort_values('yil')
axd.plot(p7.yil, p7.sigma_epistemik, color='#1f78b4', lw=0.8)
axd.set_xlabel('Year'); axd.set_ylabel('Epistemic s.d. (log ring width)')
axd2 = axd.twinx(); axd2.fill_between(p7.yil, p7.n / 1e3, color='#dddddd', lw=0, zorder=0); axd.set_zorder(axd2.get_zorder() + 1); axd.patch.set_visible(False)
axd2.set_ylabel('Rings per year (thousands)', color='#888888'); axd2.tick_params(axis='y', colors='#888888'); axd2.spines['right'].set_visible(True)
harf(axd, 'd', x=-0.36); src6.append(p7.assign(panel='d'))
kaydet(fig, 'Fig6', pd.concat(src6, ignore_index=True))

In [ ]:
# --- Extended Data Fig. 2 | Architecture search ---
if P2 is not None:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(CIFT * 0.75, 0.30 * CIFT))
    for k_, m in zip(sorted(P2.katman.unique()), ['o', 's', '^', 'D', 'v', 'P']):
        x = P2[P2.katman == k_].sort_values('ag_param')
        a1.plot(x.ag_param, x.en_iyi_dogrulama, marker=m, ms=2.5, lw=0.6, label=f'{k_} layer' + ('s' if k_ > 1 else ''))
        a2.plot(x.ag_param, x.r2_log, marker=m, ms=2.5, lw=0.6)
    s = P2[(P2.gizli == SECIM['gizli']) & (P2.katman == SECIM['katman'])]
    for ax_, c in [(a1, 'en_iyi_dogrulama'), (a2, 'r2_log')]:
        ax_.set_xscale('log'); ax_.set_xlabel('Parameters of the site network')
        if len(s): ax_.scatter(s.ag_param, s[c], s=40, facecolors='none', edgecolors='k', lw=0.7, zorder=5)
    a1.set_ylabel('Early-stopping NLL (training years)'); a2.set_ylabel('R² of log ring width (withheld years)')
    a1.legend(loc='upper right', fontsize=5); harf(a1, 'a'); harf(a2, 'b')
    fig.tight_layout(); kaydet(fig, 'ED2', P2[['katman', 'gizli', 'ag_param', 'en_iyi_dogrulama', 'r2_log', 'blok_ici_r_medyan']])

In [ ]:
# --- Extended Data Fig. 3 | Century trend of residuals ---
fig, (a1, a2) = plt.subplots(1, 2, figsize=(CIFT * 0.75, 0.30 * CIFT)); src = []
for m, rk, ad in [(ANA, '#1f78b4', 'With climate'), (IKLIMSIZ, '#999999', 'Without climate')]:
    p = P7[(P7.model == m) & (P7.n >= 1000)].sort_values('yil')
    a1.plot(p.yil, p.gozlenen - p.tahmin, color=rk, lw=0.8, label=ad); src.append(p.assign(panel='a'))
a1.axhline(0, color='k', lw=0.4); a1.set_xlabel('Year'); a1.set_ylabel('Mean residual, observed − predicted (log)'); a1.legend(); harf(a1, 'a')
if Q2y is not None:
    for i, (g, x) in enumerate(Q2y[Q2y['size'] >= 1000].groupby('series_age', sort=False)):
        a2.plot(x.decade + 5, x['mean'], marker='o', ms=2, lw=0.8, label=f'{g} yr')
    a2.axhline(0, color='k', lw=0.4); a2.set_xlabel('Decade'); a2.set_ylabel('Mean residual (log)')
    a2.legend(title='Years since first ring', title_fontsize=5); src.append(Q2y.assign(panel='b'))
harf(a2, 'b'); fig.tight_layout(); kaydet(fig, 'ED3', pd.concat(src, ignore_index=True))

In [ ]:
# --- Extended Data Fig. 4 | Ring-based estimate of growth change through extreme years ---
if HF is not None and HS is not None:
    fig = plt.figure(figsize=(CIFT, 0.36 * CIFT)); src = []
    a1 = fig.add_axes([0.06, 0.18, 0.22, 0.72])
    sic = HF[(HF.olay.str.startswith('Sıcak')) & (HF.donem == '1990–2019')]
    g3 = ['Tüm alanlar', 'En soğuk %25', 'En kurak %25']
    for i, g in enumerate(g3):
        r = sic[sic.grup == g]
        if not len(r): continue
        r = r.iloc[0]; a1.bar(i - 0.18, r.olgusal_10_yilda, 0.34, color='#b2182b', lw=0); a1.bar(i + 0.18, r.karsi_olgusal_10_yilda, 0.34, color='#92c5de', lw=0)
        src.append(dict(panel='a', group=tr2en(g), observed_per_decade=r.olgusal_10_yilda, counterfactual_per_decade=r.karsi_olgusal_10_yilda))
    a1.set_xticks(range(3)); a1.set_xticklabels([etiket(tr2en(g)) for g in g3], fontsize=5); a1.set_ylabel('Hot growing seasons per decade, 1990–2019')
    a1.legend(handles=[Patch(color='#b2182b', label='Observed climate'), Patch(color='#92c5de', label='Without climate change')], loc='lower left', bbox_to_anchor=(0, 1.0))
    harf(a1, 'a', x=-0.25)
    a2 = fig.add_axes([0.42, 0.18, 0.30, 0.72])
    sp = HS.secim.unique(); ad_sp = {s_: s_ for s_ in sp}
    ad_sp.update({sp[0]: 'Main (indicators, lag 0–1)'})
    EN = {'Gecikme 0–3 yıl': 'Lags 0–3', 'Sıcak×kurak etkileşimi': 'Hot × dry interaction', 'Etkileşim + gecikme 0–3': 'Interaction + lags 0–3',
          '"Sürekli iklim (şiddet), gecikme 0–1"': 'Continuous anomalies, lag 0–1', 'Sürekli iklim (şiddet), gecikme 0–1': 'Continuous anomalies, lag 0–1',
          'Sürekli iklim, gecikme 0–3': 'Continuous anomalies, lags 0–3', 'Olay etkisi yalnız 1902–1960': 'Responses from 1902–1960',
          'Olay etkisi yalnız 1961–2019 (ayrı kronoloji)': 'Responses from 1961–2019'}
    ad_sp.update({k_: v for k_, v in EN.items() if k_ in ad_sp})
    for j, (g, rk, mk) in enumerate([('En kurak %25', RENK['Driest 25%'], 'o'), ('Tüm alanlar', RENK['All sites'], 's'), ('En soğuk %25', RENK['Coldest 25%'], 'D')]):
        for i, s_ in enumerate(sp):
            r = HS[(HS.secim == s_) & (HS.grup == g)]
            if not len(r): continue
            r = r.iloc[0]; y = len(sp) - 1 - i + (j - 1) * 0.22
            a2.errorbar(r.toplam_yuzde_yil, y, xerr=[[r.toplam_yuzde_yil - r.blok_alt95], [r.blok_ust95 - r.toplam_yuzde_yil]], fmt=mk, ms=2.2,
                        color=rk, elinewidth=0.6, capsize=0)
            src.append(dict(panel='b', specification=ad_sp[s_], group=tr2en(g), estimate=r.toplam_yuzde_yil, lo95=r.blok_alt95, hi95=r.blok_ust95))
    a2.set_yticks(range(len(sp))[::-1]); a2.set_yticklabels([ad_sp[s_] for s_ in sp], fontsize=5); a2.axvline(0, color='k', lw=0.4)
    a2.set_xlabel('Growth change through extreme years (% per year)')
    a2.legend(handles=[Line2D([], [], marker=m, ls='', color=c, ms=3, label=l) for l, c, m in
                       [('Driest quarter', RENK['Driest 25%'], 'o'), ('All sites', RENK['All sites'], 's'), ('Coldest quarter', RENK['Coldest 25%'], 'D')]],
              loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3)
    harf(a2, 'b', x=-0.52)
    if HK is not None:
        a3 = fig.add_axes([0.82, 0.18, 0.17, 0.72])
        for j, (g, rk, mk) in enumerate([('En kurak %25', RENK['Driest 25%'], 'o'), ('Tüm alanlar', RENK['All sites'], 's'), ('En soğuk %25', RENK['Coldest 25%'], 'D')]):
            x = HK[HK.grup == g].sort_values('blok_derece')
            a3.errorbar(np.arange(len(x)) + (j - 1) * 0.18, x.ort_yuzde, yerr=[x.ort_yuzde - x.alt95, x.ust95 - x.ort_yuzde], fmt=mk, ms=2.2,
                        color=rk, elinewidth=0.6, capsize=0)
            src += [dict(panel='c', group=tr2en(g), block_deg=b_, estimate=e_, lo95=l_, hi95=h_) for b_, e_, l_, h_ in zip(x.blok_derece, x.ort_yuzde, x.alt95, x.ust95)]
        a3.set_xticks(range(3)); a3.set_xticklabels(['5°', '10°', '20°']); a3.axhline(0, color='k', lw=0.4)
        a3.set_xlabel('Spatial block size'); a3.set_ylabel('Growth change (% per year)'); harf(a3, 'c', x=-0.45)
    kaydet(fig, 'ED4', pd.DataFrame(src))

In [ ]:
# --- Extended Data Fig. 5 | Attribution by genus: network and rings ---
gen = Q3g[(Q3g.period == DON) & (Q3g.channel == 'total') & Q3g.group.str.startswith('Genus: ') & (Q3g.n_sites >= 40)].sort_values('median_pct')
fig, ax = plt.subplots(figsize=(TEK, 0.10 * TEK + 0.045 * TEK * len(gen))); src = []
for y, (_, r) in enumerate(gen.iterrows()):
    ax.errorbar(r.median_pct, y + 0.15, xerr=[[r.median_pct - r.boot_lo95], [r.boot_hi95 - r.median_pct]], fmt='o', ms=2.6, color='#1f78b4', elinewidth=0.6)
    src.append(dict(group=r.group, source='network', estimate=r.median_pct, lo95=r.boot_lo95, hi95=r.boot_hi95, n_sites=r.n_sites))
    if HB is not None:
        h = HB[(HB.secim == 'Ana') & (HB.group == r.group)]
        if len(h):
            h = h.iloc[0]
            ax.errorbar(h.toplam_yuzde_yil, y - 0.15, xerr=[[h.toplam_yuzde_yil - h.blok_alt95], [h.blok_ust95 - h.toplam_yuzde_yil]], fmt='D', ms=2.2,
                        color='#e66101', elinewidth=0.6)
            src.append(dict(group=r.group, source='rings (model-free)', estimate=h.toplam_yuzde_yil, lo95=h.blok_alt95, hi95=h.blok_ust95, n_sites=h.n_alan))
    ax.text(1.01, y, f'{int(r.n_sites)}', transform=ax.get_yaxis_transform(), fontsize=4.8, va='center', color='#666666')
ax.set_yticks(range(len(gen))); ax.set_yticklabels([etiket(g) for g in gen.group], fontstyle='italic'); ax.axvline(0, color='k', lw=0.4)
ax.set_xlabel('Growth change, 1990–2019 (% per year)')
ax.legend(handles=[Line2D([], [], marker='o', color='#1f78b4', ls='', ms=3, label='Network (20 members)'),
                   Line2D([], [], marker='D', color='#e66101', ls='', ms=2.6, label='Rings (no network)')], loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2)
kaydet(fig, 'ED5', pd.DataFrame(src))

In [ ]:
print(sorted(os.listdir(CIKTI)))
print('==== DEFTER 22 TAMAM ====')